## Import

In [4]:
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Dataset

In [5]:
from datasets import NonlinearGaussian, MoG

n, d = 10000, 64                                 # < higher d, higher MI
true_rho = 0.7                                   # < higher rho, higher MI
case = 'MoG'                                      # < choose between ['1a', '1b', '2', '3a', '3b', '3c', 'MoG']

if case != 'MoG':
    dataset = NonlinearGaussian.NonlinearGaussian(n_samples=n, n_dims=d, rho=true_rho, mu=0, case=case)
    X0, Y0 = dataset.sample_data(n_samples = n)
    X, Y = dataset.transformation(X0, Y0)
    MI = dataset.true_mutual_info()              # we know GT MI
else:
    dataset = MoG.MoG(n_samples=n, n_dims=d, K=5, shifts=[-0.2, -0.1, 0, 0.3, 0.4], rhos=[-0.3, 0.5, 0.2, 0.4, 0.9])
    X, Y = dataset.sample_data(n_samples = n)
    MI = dataset.empirical_mutual_info()         # MI by MC estimate

X, Y = X.to(device), Y.to(device)
Z = torch.cat([X, Y], dim=1)
T = torch.ones(n, 2).to(device)

print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 6.8999606299636165


## Hyperaparams

In [6]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## MI estimate

In [9]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.empirical_mutual_info())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.479509949684143 loss val= 1.2452069520950317 best val loss= 1.2452069520950317 best t= 0
finished: t= 76 loss= 0.5148487687110901 loss val= 0.6405341625213623 best val loss= 0.4994521737098694 best t= 71
finished: t= 152 loss= 0.512866199016571 loss val= 0.6307941675186157 best val loss= 0.49563345313072205 best t= 90
finished: t= 228 loss= 0.5136118531227112 loss val= 0.635785698890686 best val loss= 0.4899367690086365 best t= 182
finished: t= 304 loss= 0.5042958855628967 loss val= 0.5106587409973145 best val loss= 0.48198288679122925 best t= 232
finished: t= 380 loss= 0.7278301119804382 loss val= 0.6459895372390747 best val loss= 0.479489803314209 best t= 352
finished: t= 456 loss= 0.722673237323761 loss val= 0.5053110122680664 best val loss= 0.479489803314209 best t= 352
finished: t= 532 loss= 0.4804435670375824 loss val= 0.7473568916320801 best val loss= 0.479489803314209 best t= 352
finished: t= 608 loss= 0.7573482394218445 loss val= 0.7388638257980347 best 

In [ ]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.empirical_mutual_info())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.021398521959781647 loss val= 0.022172579541802406 best val loss= 0.022172579541802406 best t= 0
finished: t= 76 loss= -2.8781561851501465 loss val= -2.3732728958129883 best val loss= -2.3736562728881836 best t= 61
finished: t= 152 loss= -2.908348560333252 loss val= -2.230228900909424 best val loss= -2.472731351852417 best t= 91
finished: t= 228 loss= -3.0624022483825684 loss val= -2.0378360748291016 best val loss= -2.472731351852417 best t= 91


true MI: 6.901847883968588
est MI: 3.0364580154418945


In [27]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.empirical_mutual_info())
print('est MI:', estimator.MI(X, Y))


K components= 5 copula transform= True
nde type: FM
finished: t= 0 loss= 2.114959478378296 loss val= 2.049887180328369 best val loss= 2.049887180328369 best t= 0
finished: t= 126 loss= 1.6088385581970215 loss val= 1.6905312538146973 best val loss= 1.6486761569976807 best t= 49


finished: t= 0 loss= 2.0631961822509766 loss val= 2.0576367378234863 best val loss= 2.0576367378234863 best t= 0
finished: t= 126 loss= 1.635003685951233 loss val= 1.700838327407837 best val loss= 1.6473727226257324 best t= 69


finished: t= 0 loss= 479.6342468261719 loss val= 483.84747314453125 best val loss= 483.84747314453125 best t= 0
finished: t= 101 loss= 84.66829681396484 loss val= 85.73126220703125 best val loss= 85.72859191894531 best t= 100
finished: t= 202 loss= 84.57867431640625 loss val= 85.7474365234375 best val loss= 85.7264404296875 best t= 135


K components= 5 finished
true MI: 6.890926682435426
est MI: 5.652958393096924
